In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [2]:
def search(query: str) -> list[dict]:
    """
    Search the documentation database for relevant results based on a query string.

    Args:
        query (str): The search query to look up in the index.

    Returns:
        list[dict]: A list of search result objects returned by the index.
    """
    results = index.search(
        query=query,
        num_results=5
    )
    return results

def add_entry(filename, title, description, content):
    """
    Add a new documentation entry to the index.

    Args:
        filename (str): The source filename associated with the entry.
        title (str): The title of the documentation entry.
        description (str): A short description summarizing the entry.
        content (str): The full content of the documentation entry.

    Returns:
        str: A status message indicating the result of the operation ("OK").
    """
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [3]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.add_tool(add_entry)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    list[dict]: A list of search result objects returned by the index.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filenam

In [4]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [6]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [7]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [8]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


Hasta aqui teniamos el ejemplo anterior, ahora vamos a meter las tools en una clase

In [9]:
class SearchIndexTools:
    def __init__(self, index):
        self.index = index

    def search(self, query: str) -> list[dict]:
        """
        Search the documentation database for relevant results based on a query string.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            list[dict]: A list of search result objects returned by the index.
        """
        results = self.index.search(
            query=query,
            num_results=5
        )
        return results

    def add_entry(self, filename, title, description, content):
        """
        Add a new documentation entry to the index.

        Args:
            filename (str): The source filename associated with the entry.
            title (str): The title of the documentation entry.
            description (str): A short description summarizing the entry.
            content (str): The full content of the documentation entry.

        Returns:
            str: A status message indicating the result of the operation ("OK").
        """
        entry = {
            'start': 0,
            'content': content,
            'title': title,
            'description': description,
            'filename': filename,
        }
        self.index.append(entry)
        return "OK"

Ahora el indice se pasa en el constructor y la clase no tiene dependencisa

In [10]:
search_tools_object = SearchIndexTools(index)
results = search_tools_object.search('dashboard')
results

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the

In [11]:
agent_tools = Tools()
agent_tools.add_tool(search_tools_object.search)
agent_tools.add_tool(search_tools_object.add_entry)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    list[dict]: A list of search result objects returned by the index.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filenam

Pero hay un atajo para añadir todos los metodos de una clase

In [12]:
agent_tools = Tools()
agent_tools.add_tools(search_tools_object)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title': {'type': 'string', 'description': 'title parameter'},
    'description': {'type': 'string', 'description': 'description parameter'},
    'content': {'type': 'string', 'description': 'content parameter'}},
   'required': ['filename', 'title', 'description', 'content'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database f

In [13]:
agent_tools = Tools()
agent_tools.add_tools(search_tools_object)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title': {'type': 'string', 'description': 'title parameter'},
    'description': {'type': 'string', 'description': 'description parameter'},
    'content': {'type': 'string', 'description': 'content parameter'}},
   'required': ['filename', 'title', 'description', 'content'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database f

Ahora hemos movido la clase con las herramientas a search_tools.py para importarla y usarlas

In [14]:
import search_tools

search_tools_object = search_tools.SearchIndexTools(index)
result = search_tools_object.search('dashboard')
result

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the